# 阶段六：Backpressure

使用有界 `asyncio.Queue` 和 `Semaphore` 限制排队任务与同时运行的工具数量。

In [ ]:
import asyncio

async def bounded_tool(name: str, active: list[int], peak: list[int]) -> str:
    """模拟一个受并发限制的异步工具。

    参数:
        name: 工具名称。
        active: 当前活动任务数量的可变记录。
        peak: 记录观察到的最大活动任务数。

    返回:
        工具完成后的结果。
    """
    active[0] += 1
    peak[0] = max(peak[0], active[0])
    try:
        await asyncio.sleep(0.01)
        return f"{name}: done"
    finally:
        active[0] -= 1

async def run_bounded(names: list[str], queue_size: int, concurrency: int) -> tuple[list[str], int]:
    """用有界队列和固定消费者处理任务。

    参数:
        names: 待处理工具名称列表。
        queue_size: 队列最多缓存的任务数量。
        concurrency: 同时运行的最大任务数量。

    返回:
        结果列表和观察到的最大并发数。
    """
    queue: asyncio.Queue[tuple[int, str]] = asyncio.Queue(maxsize=queue_size)
    results = [""] * len(names)
    active, peak = [0], [0]

    async def consume() -> None:
        while True:
            item = await queue.get()
            if item is None:
                queue.task_done()
                return
            index, name = item
            results[index] = await bounded_tool(name, active, peak)
            queue.task_done()

    workers = [asyncio.create_task(consume()) for _ in range(concurrency)]
    for index, name in enumerate(names):
        await queue.put((index, name))
    await queue.join()
    for _ in workers:
        await queue.put(None)
    await asyncio.gather(*workers)
    return results, peak[0]

async def langchain_async_example() -> None:
    """使用 LangChain `abatch` 的 `max_concurrency` 限制并发。"""
    from langchain_core.runnables import RunnableLambda
    active, peak = [0], [0]
    async def run(name: str) -> str:
        return await bounded_tool(name, active, peak)
    runnable = RunnableLambda(func=lambda name: name, afunc=run)
    results = await runnable.abatch(["a", "b", "c", "d"], config={"max_concurrency": 2})
    assert peak[0] <= 2
    print("LangChain Async:", results, "peak=", peak[0])

async def main() -> None:
    """验证原生有界调度和 LangChain 并发上限。"""
    results, peak = await run_bounded([f"tool-{i}" for i in range(6)], queue_size=2, concurrency=3)
    assert len(results) == 6
    assert peak <= 3
    print("native Python peak=", peak)
    await langchain_async_example()

await main()


In [ ]:
from fastapi import FastAPI

app = FastAPI()

@app.post("/phase6/tools")
async def run_phase6_tools() -> list[str]:
    """提供 HTTP 入口，调用有界工具调度器。

    返回:
        所有工具的执行结果列表。
    """
    results, peak = await run_bounded([f"tool-{i}" for i in range(6)], queue_size=2, concurrency=3)
    assert peak <= 3
    return results

assert any(route.path == "/phase6/tools" for route in app.routes)
print("FastAPI route registered")
